# German Podcast to English Podcast (Runpod / RTX PRO 6000, Hugging Face only)

This notebook runs end-to-end on a Runpod instance with 1 RTX PRO 6000 (96 GB VRAM):
1. Read a German MP3 podcast
2. Transcribe with diarisation
3. Translate to English (batched via vLLM)
4. Generate a multi-speaker English podcast
5. Save the translated MP3

Target: convert a 2-hour German podcast to English in under 30 minutes.


## Model selection (for RTX PRO 6000 96 GB VRAM, <30 minutes for a 2-hour podcast)

- **ASR**: `primeline/whisper-large-v3-turbo-german` (German-finetuned Whisper Turbo; 809 M params; 2.628% WER avg on CV19/MLS/Tuda-De; loaded via transformers `pipeline` with Flash Attention 2 and batch_size=32 — WhisperX is not used because it requires CTranslate2-format weights (`model.bin`) which this model does not have)
- **Diarisation**: `pyannote/speaker-diarization-community-1` via pyannote.audio 4.0.4. Audio passed as a pre-loaded waveform tensor to bypass `AudioDecoder` (torchcodec), which is incompatible with PyTorch 2.8 stable.
- **Translation**: `google/gemma-3-4b-it` via **vLLM** (Gemma 3 4B instruction-tuned; all segments translated in a single batched pass with PagedAttention; ~8 GB VRAM at bfloat16; reduces translation of a 2-hour podcast from ~10 min to under 1 min compared to sequential HuggingFace inference)
- **TTS**: `SWivid/F5-TTS` (flow-matching TTS with zero-shot voice cloning; Apache 2.0; ~6 GB VRAM; RTF 0.15; replaces XTTS-v2 whose parent company Coqui AI shut down Jan 2024)

### Alternatives considered

| Stage | Runner-up | Notes |
|-------|-----------|-------|
| ASR | `Qwen/Qwen3-ASR-1.7B` | SOTA accuracy (Jan 2026), beats whisper-large-v3 across benchmarks, built-in forced aligner, only ~5 GB VRAM. Requires new pipeline code. |
| ASR | `nvidia/canary-1b-v2` | 25 European languages, 749 RTFx, CC-BY-4.0. Requires NeMo framework. |
| Diarisation | `BUT-FIT/diarizen-wavlm-large-s80-md-v2` | Lower DER than pyannote on several benchmarks (e.g. AMI-SDM 13.9% vs 19.9%), but CC-BY-NC license and custom pipeline. |
| Translation | `google/translategemma-12b-it` | Purpose-built translation model; outperforms NLLB-200; but 12 B params is 3× slower and 3× more VRAM than the 4 B Gemma 3 model for diminishing returns on podcast-quality text. |
| Translation | `ByteDance-Seed/Seed-X-PPO-7B` | Competes with GPT-4o quality at 7B; OpenMDW (MIT-like) license; only ~14 GB VRAM. |
| TTS | `ResembleAI/chatterbox` | MIT license, beats ElevenLabs in blind tests, emotion control. |
| TTS | `nari-labs/Dia-1.6B` | Apache 2.0, purpose-built for multi-speaker dialogue with `[S1]`/`[S2]` tags. English only. |
| TTS | `FunAudioLLM/CosyVoice2-0.5B` | Apache 2.0, SOTA speaker similarity (78%), 150ms streaming latency. |

**Do NOT install torchcodec.** No wheel is compatible with PyTorch 2.8.0 stable (see LEARNINGS.md §2). Audio is pre-loaded via librosa instead.

Before running: accept model terms on Hugging Face for `pyannote/speaker-diarization-community-1` and `google/gemma-3-4b-it`, then set your read token in `HF_TOKEN`. Place your German MP3 at `INPUT_AUDIO_PATH`.


In [ ]:
# %pip -q install "pyannote.audio==4.0.4" transformers accelerate sentencepiece "f5-tts==1.1.16" "pydub==0.25.1" soundfile librosa "vllm>=0.8.0" flash-attn --no-build-isolation


In [ ]:
import gc
import os
import torch
import numpy as np
import librosa
import pandas as pd
from pathlib import Path
from pydub import AudioSegment
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, AutoTokenizer
from transformers import pipeline as hf_pipeline
from pyannote.audio import Pipeline as DiarizationPipeline
from f5_tts.api import F5TTS

HF_TOKEN = "<token>"
INPUT_AUDIO_PATH = "german_podcast.mp3"
OUTPUT_AUDIO_PATH = "translated_podcast_en.mp3"
audio_path = INPUT_AUDIO_PATH
device = "cuda"


/root/genai-explorations/.venv/lib/python3.12/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

No module named 'torchcodec'
  warnings.warn(


In [2]:
SAMPLE_RATE = 16000

# Pre-load audio once as a numpy array (16 kHz mono) for both ASR and diarization.
# Passing a pre-loaded array avoids any file-path code path in transformers/pyannote,
# bypassing torchcodec entirely (torchcodec has no working wheel for PyTorch 2.8 stable).
audio_array, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

# --- ASR ---
# WhisperX is not used: it requires CTranslate2-format weights (model.bin) but
# primeline/whisper-large-v3-turbo-german ships as .safetensors. Use transformers directly.
# Flash Attention 2 and batch_size=32 give a significant speedup on 96 GB VRAM.
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
asr_model_id = "primeline/whisper-large-v3-turbo-german"

asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    asr_model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="flash_attention_2",
)
asr_model.to(device)
processor = AutoProcessor.from_pretrained(asr_model_id)

asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=32,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

asr_result = asr_pipe(
    {"raw": audio_array, "sampling_rate": SAMPLE_RATE},
    generate_kwargs={"language": "german"},
)
asr_chunks = asr_result["chunks"]  # list of {"text": ..., "timestamp": (start, end)}

del asr_model, asr_pipe, processor
gc.collect()
torch.cuda.empty_cache()

# --- Diarization ---
# pyannote 4.x: use token= (not use_auth_token= which was removed in 4.0)
diarize_pipeline = DiarizationPipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN,
)
diarize_pipeline.to(torch.device(device))

# Pass pre-loaded waveform tensor — bypasses pyannote's AudioDecoder / torchcodec
waveform_tensor = torch.tensor(audio_array).unsqueeze(0)  # (1, samples)
diarization = diarize_pipeline({"waveform": waveform_tensor, "sample_rate": SAMPLE_RATE})

# --- Assign speakers to ASR chunks by maximum overlap ---
def get_speaker(start, end, diarization):
    overlap = {}
    # pyannote 4.x: DiarizeOutput.speaker_diarization yields (turn, speaker) 2-tuples
    # pyannote <=3.x used diarization.itertracks(yield_label=True) -> (turn, _, speaker)
    for turn, speaker in diarization.speaker_diarization:
        o = min(turn.end, end) - max(turn.start, start)
        if o > 0:
            overlap[speaker] = overlap.get(speaker, 0) + o
    return max(overlap, key=overlap.get) if overlap else "SPEAKER_00"

rows = []
for chunk in asr_chunks:
    start, end = chunk["timestamp"]
    if end is None:
        end = start + 30.0
    speaker = get_speaker(start, end, diarization)
    rows.append({"start": start, "end": end, "speaker": speaker, "text": chunk["text"].strip()})

segments_df = pd.DataFrame(rows)
segments_df["speaker"] = segments_df["speaker"].fillna("SPEAKER_00")

del diarize_pipeline, waveform_tensor, audio_array
gc.collect()
torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
/root/genai-explorations/.venv/lib/python3.12/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See ht

In [3]:
segments_df.to_csv("segments.csv", index=False)


In [5]:
# Translation via vLLM + google/gemma-3-4b-it
#
# vLLM's continuous batching and PagedAttention process all segments in a single pass,
# reducing translation of a 2-hour podcast from ~10 min (sequential HF generate) to
# under 1 min on an RTX PRO 6000.
#
# vLLM reads the HuggingFace token from the HF_TOKEN environment variable for gated
# models like google/gemma-3-4b-it — set it before constructing the LLM object.
#
# Gemma 3 4B-IT is a multimodal model (text + vision). vLLM's V1 engine runs a dummy
# forward pass through the SigLIP vision tower during KV-cache profiling, which OOMs
# or crashes on text-only workloads. Setting limit_mm_per_prompt={"image": 0} tells
# vLLM to skip multimodal profiling entirely and treat the model as text-only.

from vllm import LLM, SamplingParams

os.environ["HF_TOKEN"] = HF_TOKEN

TRANSLATION_MODEL = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL, token=HF_TOKEN)

def _make_prompt(text: str) -> str:
    messages = [{"role": "user", "content": (
        "Translate the following German text to English. "
        "Output only the English translation, nothing else.\n\n"
        f"{text}"
    )}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

llm = LLM(
    model=TRANSLATION_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    limit_mm_per_prompt={"image": 0},  # text-only: skip SigLIP vision tower profiling
)

sampling_params = SamplingParams(temperature=0, max_tokens=512)
prompts = [_make_prompt(text) for text in segments_df["text"].tolist()]
outputs = llm.generate(prompts, sampling_params)
segments_df["text_en"] = [o.outputs[0].text.strip() for o in outputs]
segments_df.to_csv("podcast_transcript_en.csv", index=False)

del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()


INFO 02-19 23:07:35 [utils.py:261] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0}, 'model': 'google/gemma-3-4b-it'}
INFO 02-19 23:07:36 [model.py:541] Resolved architecture: Gemma3ForConditionalGeneration
INFO 02-19 23:07:36 [model.py:1561] Using max model len 2048
INFO 02-19 23:07:36 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 02-19 23:07:36 [cuda.py:257] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.
INFO 02-19 23:07:36 [registry.py:143] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore_DP0 pid=3646) INFO 02-19 23:07:43 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='google/gemma-3-4b-it', speculative_config=None, tokenizer='google/gemma-3-4b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:07<00:07,  7.15s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.33s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.15s/it]
(EngineCore_DP0 pid=3646) 


(EngineCore_DP0 pid=3646) INFO 02-19 23:08:05 [default_loader.py:291] Loading weights took 16.30 seconds
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:06 [gpu_model_runner.py:4130] Model loading took 7.79 GiB memory and 17.424200 seconds
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:11 [backends.py:812] Using cache directory: /root/.cache/vllm/torch_compile_cache/e7698eab33/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:11 [backends.py:872] Dynamo bytecode transform time: 5.13 s
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:23 [backends.py:302] Cache the graph of compile range (1, 16384) for later use
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:39 [backends.py:319] Compiling a graph for compile range (1, 16384) takes 23.48 s
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:39 [monitor.py:34] torch.compile takes 28.61 s in total
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:40 [gpu_worker.py:356] Available KV cache memory: 30.02 GiB
(EngineCore_DP0 pid=3646) WARNING 

(EngineCore_DP0 pid=3646) 2026-02-19 23:08:40,802 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=3646) 2026-02-19 23:08:40,892 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 35.70it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:09<00:00,  5.57it/s]


(EngineCore_DP0 pid=3646) INFO 02-19 23:08:52 [gpu_model_runner.py:5063] Graph capturing finished in 12 secs, took -0.82 GiB
(EngineCore_DP0 pid=3646) INFO 02-19 23:08:52 [core.py:272] init engine (profile, create kv cache, warmup model) took 46.09 seconds
INFO 02-19 23:08:53 [llm.py:343] Supported tasks: ['generate']


Adding requests:   0%|          | 0/550 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/550 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[rank0]:[W219 23:09:00.324551650 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


In [2]:
segments_df = pd.read_csv("podcast_transcript_en.csv")

In [ ]:
import torchaudio
import soundfile as sf
import contextlib
from tqdm.auto import tqdm

# torchaudio >= 2.6 requires torchcodec (incompatible with PyTorch 2.8); patch with soundfile.
def _torchaudio_load_soundfile(uri, frame_offset=0, num_frames=-1, normalize=True,
                               channels_first=True, format=None, buffer_size=4096, backend=None):
    data, sr = sf.read(uri, dtype="float32", always_2d=True)
    tensor = torch.from_numpy(data.T)
    if frame_offset:
        tensor = tensor[:, frame_offset:]
    if num_frames > 0:
        tensor = tensor[:, :num_frames]
    if not channels_first:
        tensor = tensor.T
    return tensor, sr

torchaudio.load = _torchaudio_load_soundfile

work_dir = Path("translated_segments")
work_dir.mkdir(exist_ok=True)
source_audio = AudioSegment.from_file(audio_path)

# Use the longest segment (≥3 s) per speaker as the voice reference.
# Pass the existing German ASR text as ref_text — avoids calling f5tts.transcribe,
# which uses ffmpeg under the hood and fails in this environment.
MIN_REF_DURATION_S = 3.0
speaker_refs = (
    segments_df[segments_df["end"] - segments_df["start"] >= MIN_REF_DURATION_S]
    .assign(duration=lambda df: df["end"] - df["start"])
    .sort_values("duration", ascending=False)
    .drop_duplicates("speaker")[["speaker", "start", "end", "text"]]
)
missing = set(segments_df["speaker"].unique()) - set(speaker_refs["speaker"])
if missing:
    speaker_refs = pd.concat([speaker_refs,
        segments_df[segments_df["speaker"].isin(missing)]
        .assign(duration=lambda df: df["end"] - df["start"])
        .sort_values("duration", ascending=False)
        .drop_duplicates("speaker")[["speaker", "start", "end", "text"]]
    ], ignore_index=True)

speaker_refs["ref_path"] = [work_dir / f"{r}_ref.wav" for r in speaker_refs["speaker"]]
for row in speaker_refs.itertuples(index=False):
    source_audio[int(row.start * 1000):int(row.end * 1000)].export(row.ref_path, format="wav")
speaker_ref_map = dict(zip(speaker_refs["speaker"], speaker_refs["ref_path"]))
speaker_ref_text_map = dict(zip(speaker_refs["speaker"], speaker_refs["text"]))

f5tts = F5TTS(model="F5TTS_v1_Base")

ordered_df = segments_df.sort_values("speaker").reset_index()
segment_paths = {}
with open(os.devnull, "w") as devnull:
    for row in tqdm(ordered_df.itertuples(index=False), total=len(ordered_df), desc="TTS"):
        segment_path = work_dir / f"seg_{int(row.start * 1000):09d}.wav"
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            f5tts.infer(
                ref_file=str(speaker_ref_map[row.speaker]),
                ref_text=speaker_ref_text_map[row.speaker],
                gen_text=row.text_en,
                file_wave=str(segment_path),
            )
        segment_paths[row.index] = segment_path

segment_paths = [segment_paths[i] for i in segments_df.index]


Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /root/genai-explorations/.venv/lib/python3.12/site-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /workspace/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 



TTS:   0%|          | 0/550 [00:00<?, ?it/s]

In [ ]:
translated_audio = AudioSegment.silent(duration=0)
cursor_ms = 0
for row, segment_path in zip(segments_df.itertuples(index=False), segment_paths):
    start_ms = int(row.start * 1000)
    translated_audio = translated_audio + AudioSegment.silent(duration=max(start_ms - cursor_ms, 0))
    segment_audio = AudioSegment.from_wav(segment_path)
    translated_audio = translated_audio + segment_audio
    cursor_ms = len(translated_audio)
translated_audio.export(OUTPUT_AUDIO_PATH, format="mp3")
